# ViBioMIR — crawl kho tài liệu trên Kaggle (CPU)

Notebook này chạy `mircrawl` (thư mục `mircrawl/` trong repo) cho **một shard** của danh sách 1.194.469 URL
(714.469 VI + 240.000 ZH + 240.000 PubMed). Kết quả nằm trong `/kaggle/working/out/`.

**Chuẩn bị (một lần):**
1. Tạo Kaggle Dataset (vd `vibiomir-crawl`) gồm thư mục `mircrawl/` (hoặc `mircrawl.zip`) và file
   `links_corpus_vi_en_zh.parquet`.
2. Notebook này: *Add Input* → dataset trên. *Settings* → **Internet: On**, Accelerator: **None (CPU)**.
3. (Tuỳ chọn) *Add-ons → Secrets*: `NCBI_API_KEY` (PubMed nhanh hơn 3 lần), `NCBI_EMAIL`.

**Chạy:**
- Lần đầu: `MODE = "probe"` (vài URL mỗi tên miền, ~5–10 phút) → xem bảng lỗi theo tên miền từ IP của Kaggle.
- Sau đó: `MODE = "crawl"`, mỗi notebook / tài khoản một `SHARD` khác nhau (`0/4`, `1/4`, …). Bấm
  **Save Version → Save & Run All** (chạy nền, tối đa 12 giờ; script tự dừng êm trước `TIME_BUDGET_H`).
- Chưa xong trong 1 phiên: tạo phiên mới của **cùng notebook**, *Add Input* → output của phiên trước
  (Notebook Output), chạy lại. Script tự bỏ qua URL đã xong và chép dữ liệu cũ sang output mới, nên output
  phiên cuối luôn chứa đủ dữ liệu của shard.
- Khi mọi shard xong: notebook mới (hoặc notebook này với `MODE = "merge"`), *Add Input* output của mọi shard →
  `corpus_out/corpus/*.parquet` + `status.parquet` + `report.tsv`.

In [ ]:
# ======================= CẤU HÌNH =======================
MODE = "probe"          # "probe" | "crawl" | "merge" | "report"
SHARD = "0/4"           # i/N — mỗi notebook / tài khoản một i khác nhau, N giống nhau ở mọi nơi
TIME_BUDGET_H = 11.3    # phiên CPU Kaggle tối đa 12 giờ; chừa thời gian ghi file + báo cáo
PROBE_PER_HOST = 5      # MODE="probe": số URL mỗi tên miền (PubMed ×20)
EXTRA_ARGS = []         # vd ["--skip-hosts", "familydoctor.com.cn,zysjonline.com"] hoặc ["--no-raw"]
OUT = "/kaggle/working/out" if MODE != "merge" else "/kaggle/working/corpus_out"

In [ ]:
# Thư viện: curl_cffi (giả lập vân tay TLS của Chrome → ít bị 403), trafilatura (trích nội dung trang không có quy tắc riêng)
!pip install -q "curl_cffi>=0.7" "trafilatura>=1.12"

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

IN = "/kaggle/input"
code_init = glob.glob(f"{IN}/**/mircrawl/__init__.py", recursive=True)
CODE_ROOT = "/tmp/mircrawl_code"
shutil.rmtree(CODE_ROOT, ignore_errors=True)
if code_init:
    shutil.copytree(os.path.dirname(code_init[0]), f"{CODE_ROOT}/mircrawl")
else:
    zips = glob.glob(f"{IN}/**/mircrawl.zip", recursive=True)
    assert zips, "Không thấy thư mục mircrawl/ hoặc mircrawl.zip trong /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(CODE_ROOT)
LINKS = (glob.glob(f"{IN}/**/links_corpus_vi_en_zh.parquet", recursive=True) or [None])[0]
assert LINKS, "Không thấy links_corpus_vi_en_zh.parquet trong /kaggle/input"

# Output của các phiên / shard trước (Add Input → Notebook Output): mọi thư mục cha của pages/*.parquet
PREV = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{IN}/**/pages/*.parquet", recursive=True)})

env = dict(os.environ, PYTHONPATH=CODE_ROOT, PYTHONUNBUFFERED="1")
try:
    from kaggle_secrets import UserSecretsClient
    for k in ("NCBI_API_KEY", "NCBI_EMAIL"):
        try:
            env[k] = UserSecretsClient().get_secret(k)
        except Exception:
            pass
except ImportError:
    pass
sys.path.insert(0, CODE_ROOT)
import mircrawl
print("mircrawl", mircrawl.__version__, "xv", mircrawl.XV, "| links:", LINKS)
print("CPU:", os.cpu_count(), "| NCBI_API_KEY:", "có" if env.get("NCBI_API_KEY") else "không")
print("Output phiên trước:", *PREV, sep="\n  ")

In [ ]:
def run(args):
    cmd = [sys.executable, "-m", "mircrawl"] + [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=CODE_ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

if MODE == "probe":
    rc = run(["probe", "--links", LINKS, "--out", OUT, "--probe", PROBE_PER_HOST, *EXTRA_ARGS])
elif MODE == "crawl":
    rc = run(["crawl", "--links", LINKS, "--out", OUT, "--shard", SHARD, "--time-budget", TIME_BUDGET_H,
              "--resume", *PREV, *EXTRA_ARGS])
    run(["report", "--inputs", OUT, "--links", LINKS, "--shard", SHARD, "--out", OUT])
elif MODE == "report":
    rc = run(["report", "--inputs", *PREV, "--links", LINKS, "--out", "/kaggle/working"])
elif MODE == "merge":
    rc = run(["merge", "--inputs", *PREV, "--out", OUT, "--links", LINKS, *EXTRA_ARGS])
print("exit code:", rc)

### Xem nhanh kết quả
`pages/*.parquet`: mỗi URL đã thử một dòng — `status` (ok / empty / notice / gone / http_xxx / blocked_* / err_*),
`title`, `text` (văn bản có cấu trúc: `#` tiêu đề mục, `- ` danh sách, `a | b` hàng bảng, `---` ranh giới hỏi–đáp),
`lang`, `doc_type`, `published`, `author`, `section`, `keywords`, `extra` (JSON: người trả lời, PMID, MeSH, DOI…).

In [ ]:
import pandas as pd
src = f"{OUT}/corpus" if MODE == "merge" else f"{OUT}/pages"
df = pd.read_parquet(src, columns=["id", "host", "status", "lang", "doc_type", "title", "published", "author", "n_chars", "text"])
print(df.status.value_counts().head(15))
for _, r in df[df.status == "ok"].groupby("host").head(1).head(12).iterrows():
    print(f"\n== {r.host} [{r.lang} {r.doc_type}] {r.published} | {r.author[:30]} | {r.title[:80]}")
    print("\n".join(r.text.split("\n")[:4])[:400])